In [1]:
!pip install chromadb sentence-transformers ollama

In [51]:
!pip install openai

In [52]:
from google import genai
from openai import OpenAI
from sentence_transformers import SentenceTransformer


In [53]:
model = SentenceTransformer("BAAI/bge-m3")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 21118.97it/s]


In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="chroma_db")

collection = chroma_client.get_collection("arabic_poetry")

In [ ]:

client = OpenAI(
    api_key="API KEY"
)

In [129]:
poet = " جرير"
theme = "هجاء"
topic = "الكذب"


theme_filter = f"قصائد {theme}"

In [130]:
query = f"قصائد {poet} في {theme}"

query_embedding = model.encode(query).tolist()

In [131]:
results = collection.query(
    query_embeddings=[query_embedding],
    where={
        "$and": [
            {"poet": poet},
            {"theme": theme_filter}
        ]
    },
    n_results=3
)

In [132]:
titles = []

for meta in results["metadatas"][0]:
    title = meta["title"]

    if title not in titles:
        titles.append(title)

print(f"Retrieved {len(titles)} unique poems")

Retrieved 0 unique poems


In [133]:
examples = ""

for i, doc in enumerate(results["documents"][0], 1):

    # Extract only the poem part
    poem = doc.split("Poem:")[-1].strip()

    # Keep only the first 12 verses
    verses = poem.split("\n")
    poem = "\n".join(verses[:12])

    examples += f"""
Example {i}

{poem}

----------------------------------------

"""

In [134]:
print(examples[:4000])

Create The Prompt

In [135]:
prompt = f"""
You are an expert Arabic poet.

Study the STYLE of the following poems.

Do NOT copy any verse.
Do NOT repeat any sentence.
Do NOT continue the examples.

Learn only:
- vocabulary
- rhythm
- imagery
- expressions

Style Examples:

{examples}

===================================

Now write a completely NEW poem.

Topic: {topic}
Poet style: {poet}
Theme: {theme_filter}

Requirements:
- Around 10 verses.
- Classical Arabic.
- Original wording.
- Do not copy any line from the examples.

Return ONLY the poem.
"""

Generate The Poem

In [136]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.9
)

print(response.choices[0].message.content)

كذبتَ فما للكذب في الناس مفرُّ  
هو السُمُّ الذي إذا انتشر قد استشرى  

تظنُّ الدنيا لك بحسنٍ تزهو بهِ  
وغداً ترى النجوم قد أظلمت واستثرت  

كذبٌ منك سهمٌ في القلبِ قد صدَّقوهُ  
وفازَ به السفهاءُ، فما خاتمتهُ سُرَّى  

لا يدوم في قلوب الرجال ذاك الكذبُ  
فأنهك الأيام بالزيف ما يُحتسبُ  

تَعلَّقَ بالزورِ فغداَ لسانكَ سُمّاً  
لا يُشفى من نفْسِهِ جُرحُهُ ولا يُرتجى  

فاحذر من كذبٍ ينحت في وجه الدنيا  
كما تنحت الصقور الجبال فتدمَّرتْ!


# Results

Below are 10 examples generated by the system using different poets, themes, and topics.

Example 1

Poet: المتنبي
Style: قصائد حكمة
Topic: العلم

Generated Poem:

إذا ما ارتقى العلمُ في نفوسِ فتىً *** تاه عن دنسِ الجهلِ كلُّ مَكْرَهِ

وصارَ الأفقُ لهُ منارًا يُضيءُ *** والصبرُ مفتاحُهُ في كلِّ مَفْتَهِ

لا يَذلُّ من حملَ علْمًا على الأعناقِ *** ولو جَرى الدهرُ عليهِ نَحرَهُ وهَتَكِ

يَركبُ المجدَ بحُصانهِ الحكمةِ *** ويرفَعُ رايةَ الفهمِ فوقَ القممِ

ليسَ يطلبُ العلمَ إلا من غوى بِهِ *** أو مُحبٍّ لهُ، فذاك هو الحِكَمُ

فكم عَلَّمَ الدهرُ من جاهلٍ صَبُرًا *** حتى دَجَّنَ في قلبهِ بَذورَهُ الثِّمَرِ

فالعلمُ للبشرِ كالندى للفجرِ *** يوقظُ العقلَ من غفلةِ السِّترِ

فلا تَصغرَنَّ من العلمِ هِمَّةً *** فالعلمُ مرسى النفسِ في السفرِ

وما الحياةُ إلا جَسرٌ عَابِرٌ *** والعلمُ لهُ السُلوكُ والبَصَرُ

Example 2

Poet: المتنبي
Style: قصائد حكمة
Topic: الصبر

Generated Poem:

إذا صبر المرء على ما يكتوي بهِ *** نالَ من صيفِ الحياةِ بردَ الظلالِ  
والدهرُ لا يلينُ للضعيفِ جفاهُ *** كالصخرِ إذ تقسى بأغلالِ الجبالِ  
فالصبرُ مفتاحُ فرجٍ لا يُدرَكُ *** إلا لمن أُشفقَ على نفسه من الأحوالِ  
تهونُ المصائبُ في صدرِ الصابرينَ *** كالندى بعد ظمأٍ في زهرِ الجبالِ  
ولينُ النفسِ في الضيقِ دليلُ الثباتِ *** وسرُّ نجاحِ العزائمِ في المثابرةِ  
من صبرَ على القهرِ صارَ فوقَ الناسِ *** كالنجمِ في وسطِ ظلماءِ الليالي  
لا يقاسُ الإنسانُ بقوةِ يدهِ *** بل بحجمِ ما يملكُ من التجاهلِ  
فإن عشتَ في دنيا الناسِ صابراً *** فلا تحسبنَّ في الصبرِ من ضعفِ المآلِ  
كلُّ مشقةٍ لها في الفجرِ أملٌ *** ومن ضاقت بهِ الأرضُ، ضاقت بهُ الآمالُ  
فاصبرْ إنما الصبرُ تاجُ المجدِ *** وقنديلُ العقلِ في زمنِ الظلالِ

Example 3

Poet: الإمام الشافعي
Style: قصائد نصيحة
Topic: الأخلاق

Generated Poem:

إذا الأخلاقُ جُناحُ الإنسانِ بهِ  
طارَ في سُموِّ العلياءِ بلا رهَبِ  

فاحفظْ فؤادَكَ مِن دنسِ الرذائلِ  
وازرعْ الخيرَ نابلاً في حقلِكَ الطَّيبِ  

لا يُزهي الكرمُ إلا بخلقٍ حَسَنٍ  
ولا تطيبُ الحياةُ دون ودادٍ عميقِ  

أنَّ النفسَ كالمرآةِ إن صَفَّوْها  
عكستْ ضياءَ الحقِّ في أسمى الأصلابِ  

فكن للخلقِ كالندى في الصباحِ نافعاً  
وأضاءْ لغيركَ دروبَ الخيرِ القريبِ  

لا تقلْ للناسِ: إنّي في الأصلِ طيبٌ  
بل كنْ في القولِ والفعلِ كالذهبِ الثمينِ  

إذا اخْتَلَفتَ في الحياةِ فلا تمِلَّ  
فالأخلاقُ نبراسُ المُلتقى والحُبيبِ

Example 4

Poet: أحمد شوقي
Style: قصائد وطنية
Topic: الوطن

Generated Poem:

يا وطنَ المجدِ، يا مهدَ الأبطالِ  
فيك الحُمى والندى، وعطرُ الجمالِ  

تحتَ ظلكَ يزهو القلبُ ويطيبُ  
وفي ثراكَ تُروى قصصُ الأجيالِ  

نُسجتْ فيك الأملُ كريحانٍ نضرٍ  
ينثرُ الوفاءَ بقلوبِ الأمثالِ  

يا درةَ الدُّنيا بينَ السهولِ والماءِ  
يا صوتَ الحسنِ، ويا نبعَ الكمالِ  

منك نرتقي، وفيك السلامُ يُشرقُ  
وفي رحابكَ تمتدُّ دروبُ الأمانِ  

لا ننسى دماءَ الأبطالِ الزكيةِ  
لوطنٍ سرى في النفسِ كنجمٍ سطِعَا  

فليشهدِ الزمنُ حبنا لا يُغتالُ  
وليعشْ الوطنُ في القلبِ ذاكرَ الجمالِ

Example 5

Poet: نزار قباني
Style: قصائد غزل
Topic: الحب

Generated Poem:

يا سيدتي في عينيكِ بحرٌ من سُخْرِ الندى  
وقلبكِ نَجْمٌ يَهْدِينِي في ليل الهوى  

تُداعبُ أنفاسكِ نسيمَ نغماتِ الشوقِ  
وترسمُ لي على شفتيكِ قصائدَ العِشْقِ  

يا زهرةً تفتَّحَتْ في روابي الغرامِ  
وعطرُكِ يسري في دمي كأحلى الأَلحانِ  

في حضنكِ الدافئِ وجدتُ سَريرَ الأمانِ  
وبهفةِ القلبِ صرتُ كالغيمِ إذا انهمرْنا  

يا طيفَ الحُبِّ الذي لا يَغيبُ عن ناظري  
وصوتَكِ شذىً يُغنّي ألحانَ فؤادي  

سكنتِ الروحَ حروفاً وشغفاً وحنيناً  
وأصبحتِ في أيامي أجملَ تَغْريداً

Example 6

Poet: محمود درويش
Style: قصائد وطنية
Topic: الحرية

Generated Poem:

في ليل الغربة تضيء شمس الحرية  
كأنها وعدٌ بالمطر في أرضٍ جدباء  

تُزهر الأماني فوق رماد السجون  
وترتيل القلوب نغمات الفجر والأزاهير  

يا روحَ الوطنِ لا تَسقطي خلفَ الجدرانِ  
فالحُلمُ لا يُقيدُهُ قيدُ السُجونِ الغابرِ  

نسيرُ بخُطى الحُبِّ نحو المجدِ المسطورِ  
ونرزُعُ من الأرضِ أزهارَ الحُلمِ الناضجِ  

في عيونِ الأطفالِ تتلألأ نجومُ السلامِ  
وعلى شواطئِ الغدِ تبني الأيادي حضارةَ الأماني  

لا تُقهرُ الروحُ حين يعلو صوتُ الحريةِ  
فهي الفجرُ الذي يضيءُ دروبَ المستحيلِ

Example 7

Poet: أبو العتاهية
Style: قصائد زهد
Topic: الدنيا

Generated Poem:

يا دارَ غُرورٍ قد زالت مُنيتُها  
وجمالُ الزيتونِ فيها كذِبُ الورى  

كم من زَهْرٍ قد ذَبُلَ فكأنّه  
في الحَقْلِ لا خَضَرٌ بل سرابٌ وهْنَا  

لا تَغُرَّنكَ بَسْمَةُ الأيامِ الفانيةِ  
فمَقالُ الدهرِ لا يُبقي ولا يُبذَرُ  

أعزُّ ما في الدُنيا النفسُ الطاهرةُ  
وما المالُ إلا قيدٌ يمضي ويُهدرُ  

كم من عبدٍ جرى وراءَ الثراءِ ناسيًا  
أنّ الموتَ آتٍ لا يَرقُبُ عذرًا ولا يُفرُّ  

فأقمِ العدلَ في أيامِكَ فربّ يومٍ  
تُسألُ فيه عن كلِّ عملٍ قد جَرَى  

دعِ اللذاتِ فإنّها عَلَى حينٍ  
كغُبارِ الصحراءِ ما لبث ولا يُقرُّ  

واذكرِ الحبيبَ فقد كان الهدى نَجمًا  
يشقُّ ظُلماتِ الدهرِ بأنوارٍ سَطَرُ  

ففي زهدِ الدنيا حياةُ القلبِ والسلامُ  
وبتركِها تَطيبُ النفسُ وترتقي للسرُّ

Example 8

Poet: أبو تمام
Style: قصائد مدح
Topic: الكرم

Generated Poem:

أَيُّها السَّمْحُ حُسْنَ كَرَمٍ إن جَادَ فَلا يُضَاهى  
وَسَخَاءُهُ كَالبَحرِ في مُداهُ لا يَغَيبُ عَنا  
يُغدِقُ الأيّامَ نَفيسَةً وَيَمَنَحُ السَّرى  
كَأَنَّ السُّحُبَ العَذبةَ تَسقِطُ مِن جَنانهِ  
يَدُهُ بَذلٌ لا تَغورُ وَوجهُهُ أَفْضَلُ نُبلٍ  
يَدعو الضُّيُوفَ للضِّياءِ وَالقَلبُ مَسْكَنُ الوُدِّ  
مِنْهُ يَبلُغُ الندى وَيَزْهَرُ دَهرُ الأَخْلاقِ  
وَلا يَخشى الفَقرَ وَلا يَرعَى في قَلبِهِ نَقصٌ  
رَسَمَ الكَرَمَ بِالرِّضا بَيتًا يُشهَدُ فيهِ العِطاءُ  
فَهوَ في سُرادِقِ الكَرَمِ سَيِّدٌ لا يَزولُ أَبَداً

Example 9

Poet: عنترة بن شداد
Style: قصائد فخر
Topic: الشجاعة

Generated Poem:

يا سائلي عن صرحٍ تجلى ** في نَفوس الرجال عنترا  
شجاعةٌ ما انثنت أبدا ** على درب العزّ مددا  
لا يهابها سهم غادرٌ ** ولا تحدّت بها شدّدا  
قلبُ الفارسِ كالصخرِ صلدٌ ** إذا ما نزل الليلُ جدّدا  
تهتزّ رياح الفخر طيبةٌ ** بها الأنامُ فاحت منذرا  
غمد السيف إذا انتفضَ بسّامٌ ** في ميادين الوغى ضدّدا  
لا تستكينُ للريح الهائجِ ** ولا تغيبُ عن الكرام هدى  
تعلّمناها من جدودٍ ** بالدماء رسموا الفدا  
فجعلوا من الخوف قيدًا ** والحريّة بها ترتقى  
فكن شجاعاً بين الأنامِ ** واصنع في المجد قصدا

Example 10

Poet: جرير
Style: قصائد هجاء
Topic: الكذب

Generated Poem:
كذبتَ فما للكذب في الناس مفرُّ  
هو السُمُّ الذي إذا انتشر قد استشرى  

تظنُّ الدنيا لك بحسنٍ تزهو بهِ  
وغداً ترى النجوم قد أظلمت واستثرت  

كذبٌ منك سهمٌ في القلبِ قد صدَّقوهُ  
وفازَ به السفهاءُ، فما خاتمتهُ سُرَّى  

لا يدوم في قلوب الرجال ذاك الكذبُ  
فأنهك الأيام بالزيف ما يُحتسبُ  

تَعلَّقَ بالزورِ فغداَ لسانكَ سُمّاً  
لا يُشفى من نفْسِهِ جُرحُهُ ولا يُرتجى  

فاحذر من كذبٍ ينحت في وجه الدنيا  
كما تنحت الصقور الجبال فتدمَّرتْ!